In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import time
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# ==============================
# CONFIGURACIÓN GLOBAL
# ==============================
pyautogui.FAILSAFE = False

# Variables de Control
sensibilidad = 2
mano = 0
fc = 4
ejecutando = False
positions = []
window_size_filtro = 1000
Fs = 30.0
orden = 2
b, a = None, None

# Estado de Clicks y Scroll
FFD = 0
FFI = 0
estado_scroll = 0
valor_anterior_y = 0
tiempo_previo = time.time()
tiempos_fps = []

# Inicialización MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False, 
    max_num_hands=1, 
    min_detection_confidence=0.7
)
cap = cv2.VideoCapture()
timer = QTimer()

# ==============================
# LÓGICA DE PROCESAMIENTO
# ==============================
def actualizar_coeficientes():
    global b, a
    fs_segura = max(Fs, fc * 2.1)
    b, a = sig.butter(orden, fc/(0.5 * fs_segura), btype='low')


def suavizar_pos(x, y):
    global positions
    positions.append((x, y))
    if len(positions) > window_size_filtro: 
        positions.pop(0)
        
    if len(positions) > orden * 2:
        xs, ys = zip(*positions)
        return sig.lfilter(b, a, xs)[-1], sig.lfilter(b, a, ys)[-1]
    return x, y


def f_tecla(FF_val, boton, tecla):
    if FF_val == 0 and boton[0]: 
        pyautogui.mouseDown(button=tecla)
        return 1
        
    elif FF_val == 1 and not boton[0]: 
        pyautogui.mouseUp(button=tecla)
        return 0
        
    if boton[1]: 
        pyautogui.click(button=tecla, clicks=2)
        
    return FF_val


def mouse_ubicacion(hl):
    mouse_x, mouse_y = int(hl.landmark[0].x * 640), int(hl.landmark[0].y * 480)
    mouse_x, mouse_y = suavizar_pos((mouse_x), (mouse_y))
    
    return mouse_x, mouse_y

def mouse_gesto_operacion(hl):
    bi1 = hl.landmark[8].y > hl.landmark[5].y    
    bi2 = hl.landmark[16].y > hl.landmark[13].y
    
    bd1 = hl.landmark[12].y > hl.landmark[9].y
    bd2 = hl.landmark[20].y > hl.landmark[17].y

    return (bi1, bi2),(bd1, bd2)


def Deteccion_mano_openCV(frame):
    frame = cv2.flip(cv2.resize(frame, (640, 480)), 1)
    coord_mano = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    return coord_mano


def Deteccion_gestos(hl):
    mouse_x, mouse_y = mouse_ubicacion(hl)
    BI,BD = mouse_gesto_operacion(hl)
    scroll,estado_scroll = mouse_gesto_operacion_scroll(hl)
    
    return BI,BD,mouse_x, mouse_y,scroll,estado_scroll


FFD=FFI=0
def mouse_virtual(mouse_X ,mouse_Y , BD, BI,scroll,estado_scroll, sensibilidad,mano):
   # Mouse Move

    global  FFD,FFI
    sw, sh = pyautogui.size()
    y = int((mouse_Y - 240) * sensibilidad * sh / 480)

    if mano == 0:
        x = int((mouse_X - 320) * sensibilidad * sw / 640) 
    else:
        x = int(mouse_X * sensibilidad * sw / 640)

    try: pyautogui.moveTo(x, y, _pause=False)
    except: pass

    pyautogui.scroll(100*scroll)

    if estado_scroll == 0:
        FFD = f_tecla(FFD, BD, 'right')
        FFI = f_tecla(FFI, BI, 'left')


estado_scroll = 0
valor_anterior_y = 0
def mouse_gesto_operacion_scroll(hl):
    
    global estado_scroll, valor_anterior_y
    scroll = 0

    gesto_estado_subir = (abs(hl.landmark[8].x - hl.landmark[13].x)*640 < 30) and estado_scroll == 0
    gesto_estado_bajar = (abs(hl.landmark[5].y - hl.landmark[8].y)*480 < 50) and estado_scroll == 0
    gesto_estado_reposo = (abs(hl.landmark[0].y - hl.landmark[9].y)/5) < abs(hl.landmark[17].x - hl.landmark[5].x)

    # ESTADOS
    if(gesto_estado_subir): estado_scroll = 1
    if(gesto_estado_bajar): estado_scroll = 2
    if(gesto_estado_reposo): estado_scroll = 0


    # SALIDAS
    #SALIDA SUBIDA
    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        #print("1")
        scroll = 1

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5  and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        scroll = 0


    #SALIDA BAJADA    
    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y              
        scroll = 0

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y                
        scroll = -1
        
    return scroll,estado_scroll


def frecuencia_muestreo():
    global tiempo_previo, Fs
    t_actual = time.time()
    dt = t_actual - tiempo_previo
    tiempo_previo = t_actual
    if dt > 0: 
        tiempos_fps.append(dt)
        
    if len(tiempos_fps) > 50: 
        tiempos_fps.pop(0)

    if len(tiempos_fps) == 50:
        Fs = 1 / (sum(tiempos_fps) / 50)
        actualizar_coeficientes()
    #print(f"FPS: {Fs:.2f}", end="\r")
    
def procesar_loop():
    frecuencia_muestreo()
    
    ret, frame = cap.read()
    if not ret: 
        return

    coord_mano = Deteccion_mano_openCV(frame)

    if coord_mano.multi_hand_landmarks:
        hl = coord_mano.multi_hand_landmarks[0] # hl: hand landmark (puntos de referencia)
        
        BI,BD,mouse_X,mouse_Y,scroll,estado_scroll = Deteccion_gestos(hl)      
        mouse_virtual(mouse_X ,mouse_Y , BD, BI,scroll, estado_scroll, sensibilidad,mano)
     

# ==============================
# CONTROL DE UI Y RECURSOS
# ==============================
def apagar_recursos():
    global ejecutando
    ejecutando = False
    timer.stop()
    if cap.isOpened(): 
        cap.release()
        
    boton_inicio.setText("Iniciar")

def iniciar_programa():
    global ejecutando, tiempo_previo, positions, tiempos_fps
    if not ejecutando:
        cap.open(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            return QMessageBox.critical(None, "Error", "Cámara no disponible")
        ejecutando = True
        boton_inicio.setText("Pausar")
        timer.start(30)
        positions, tiempos_fps = [], []
        tiempo_previo = time.time()
    else:
        apagar_recursos()

def cerrar_aplicacion(event):
    apagar_recursos()
    hands.close()
    cv2.destroyAllWindows()
    event.accept()

# ==============================
# SETUP DE PYQT5 (SIN CLASE)
# ==============================
# EVITA EL ERROR DE "QApplication instance already exists"
app = QApplication.instance() or QApplication(sys.argv)

ventana = QWidget()
ventana.setWindowTitle("Control Gestual")
ventana.setFixedSize(500, 450)
ventana.closeEvent = cerrar_aplicacion # Inyectamos el cierre

layout = QVBoxLayout()
lbl_logo = QLabel()
pix = QPixmap("mouse virtual logo.jpg")
if not pix.isNull(): lbl_logo.setPixmap(pix.scaled(200, 150, Qt.KeepAspectRatio))
lbl_logo.setAlignment(Qt.AlignCenter)

boton_inicio = QPushButton("Iniciar")
boton_inicio.clicked.connect(iniciar_programa)

# Mano
radio_der = QRadioButton("Derecha")
radio_der.setChecked(True)
radio_der.toggled.connect(lambda: globals().update(mano=0 if radio_der.isChecked() else 1))

radio_izq = QRadioButton("Izquierda")

# Sliders
sld_sens = QSlider(Qt.Horizontal); sld_sens.setRange(1, 10)
sld_sens.setValue(sensibilidad)
sld_sens.valueChanged.connect(lambda v: globals().update(sensibilidad=v))

sld_fc = QSlider(Qt.Horizontal)
sld_fc.setRange(1, 10)
sld_fc.setValue(fc)
sld_fc.valueChanged.connect(lambda v: (globals().update(fc=v), actualizar_coeficientes()))

layout.addWidget(lbl_logo)
layout.addWidget(boton_inicio)

layout.addWidget(QLabel("Mano"))
layout.addWidget(radio_der)
layout.addWidget(radio_izq)

layout.addWidget(QLabel("Sensibilidad"))
layout.addWidget(sld_sens)

layout.addWidget(QLabel("Filtro (FC)"))
layout.addWidget(sld_fc)

ventana.setLayout(layout)
actualizar_coeficientes()
timer.timeout.connect(procesar_loop)

ventana.show()
app.exec_()